# Higher AUC, Wrong Reasons? Test-Time and Few-Shot Adaptation for Cross-Institutional Tuberculosis Detection with Lung-Masked Attention Auditing

**Eksperimen lengkap (Shenzhen ↔ Montgomery).** Jalankan dengan *Save Version → Save & Run All (Commit)*.

**Setting Kaggle yang wajib:**
1. *Accelerator*: **GPU T4 x1** atau **P100**.
2. *Internet*: **On** (untuk mengunduh bobot ImageNet dari timm).
3. *Add Input* → dataset **"Chest Xray Masks and Labels"** (`nikhilpandey360/chest-xray-masks-and-labels`).
   Alternatif yang juga terbaca otomatis: `kmader/pulmonary-chest-xray-abnormalities`. Boleh ditambah keduanya.

Kode mencari file secara otomatis berdasarkan pola nama `CHNCXR_xxxx_y` (Shenzhen) dan `MCUCXR_xxxx_y` (Montgomery);
digit terakhir `y` adalah label (0 = normal, 1 = TB). Masker paru dicari dari file yang path-nya mengandung kata *mask*.

**Lengan eksperimen**
| Kelompok | Metode |
|---|---|
| Batas bawah | Source-only |
| Pra-proses | CLAHE, histogram matching ke target |
| Domain generalization | MixStyle |
| Test-time / source-free | AdaBN, TENT, SHOT |
| Few-shot | fine-tuning k = 5, 10, 20 citra per kelas target |
| Batas atas | 5-fold CV di domain target |

**Audit penjelasan**: Lung Attention Ratio (LAR) Grad-CAM, *enrichment* terhadap luas paru, dan uji oklusi
(AUC saat area luar paru diblur vs saat paru diblur).

Semua keluaran tersimpan di `/kaggle/working/results/`.

In [ ]:
# ===================== 0. KONFIGURASI =====================
import os, re, glob, json, time, math, random, copy, warnings
warnings.filterwarnings("ignore")

QUICK = False            # True = uji cepat (1 seed, epoch sedikit). Untuk hasil naskah: False
INPUT_ROOT = "/kaggle/input"
OUT_DIR = "/kaggle/working/results"
PRETRAINED = True        # bobot ImageNet (butuh Internet On)
# (override untuk pengujian lokal; di Kaggle variabel ini tidak ada sehingga diabaikan)
QUICK = os.environ.get("TB_QUICK", str(QUICK)) == "True"
INPUT_ROOT = os.environ.get("TB_INPUT_ROOT", INPUT_ROOT)
OUT_DIR = os.environ.get("TB_OUT_DIR", OUT_DIR)
PRETRAINED = os.environ.get("TB_PRETRAINED", str(PRETRAINED)) == "True"

CFG = dict(
    img_size=224,
    backbones=["densenet121", "efficientnet_b0"],
    directions=[("SZ", "MC"), ("MC", "SZ")],
    arms=["baseline", "clahe", "histmatch", "mixstyle"],   # cara melatih model sumber
    seeds=[0, 1, 2, 3, 4],
    epochs=25, patience=6, batch=32, lr=3e-4, wd=1e-4,
    pool_per_class=20,           # citra target berlabel yang disisihkan untuk few-shot
    shots=[5, 10, 20],
    fs_epochs=30, fs_lr=5e-5,
    tent_steps=1,                # jumlah epoch TENT di target
    shot_epochs=5, shot_beta=0.3,
    oracle_folds=5, oracle_seeds=[0],
    cam_batch=16,
    num_workers=2,
)
if "TB_IMG" in os.environ:
    CFG["img_size"] = int(os.environ["TB_IMG"])
if QUICK:
    CFG.update(seeds=[0], epochs=3, patience=2, fs_epochs=3, shot_epochs=1,
               arms=["baseline"], oracle_folds=2)   # uji kedua backbone

os.makedirs(OUT_DIR, exist_ok=True)
print(json.dumps(CFG, indent=1))

In [ ]:
# ===================== 1. LIBRARY =====================
import subprocess, sys
try:
    import timm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm"], check=True)
    import timm
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy import stats

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__, "| timm", timm.__version__)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

In [ ]:
# ===================== 2. DATA: PENCARIAN OTOMATIS =====================
KEY_RE = re.compile(r"(CHNCXR|MCUCXR)_(\d{4})_([01])", re.I)

def discover(root):
    files = [p for p in glob.glob(os.path.join(root, "**", "*.*"), recursive=True)
             if p.lower().endswith((".png", ".jpg", ".jpeg"))]
    imgs, masks = {}, {}
    for p in files:
        m = KEY_RE.search(os.path.basename(p))
        if not m:
            continue
        key = f"{m.group(1).upper()}_{m.group(2)}_{m.group(3)}"
        # cek kata 'mask' hanya di nama file & folder induk langsung
        # (nama dataset 'chest-xray-masks-and-labels' sendiri mengandung kata 'mask')
        parts = [s.lower() for s in p.split(os.sep)[-2:]]
        if any("mask" in s for s in parts):
            masks.setdefault(key, []).append(p)
        else:
            imgs.setdefault(key, p)            # duplikat antardataset: ambil yang pertama
    rows = []
    for key, p in sorted(imgs.items()):
        site = "SZ" if key.startswith("CHN") else "MC"
        rows.append(dict(key=key, site=site, label=int(key[-1]), path=p,
                         mask_paths=masks.get(key, [])))
    return pd.DataFrame(rows)

META = discover(INPUT_ROOT)
assert len(META) > 0, "Tidak ada citra ditemukan. Cek Add Input dataset."
print(META.groupby(["site", "label"]).size().unstack())
print("Punya masker:", META.assign(m=META.mask_paths.str.len() > 0).groupby("site").m.sum().to_dict())

In [ ]:
# ===================== 3. MUAT KE MEMORI =====================
S = CFG["img_size"]

def load_gray(p):
    im = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    return cv2.resize(im, (S, S), interpolation=cv2.INTER_AREA)

def load_mask(paths):
    if not paths:
        return None
    m = np.zeros((S, S), np.uint8)
    for p in paths:                         # gabungkan masker kiri+kanan bila terpisah
        mk = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        mk = cv2.resize(mk, (S, S), interpolation=cv2.INTER_NEAREST)
        m |= (mk > 127).astype(np.uint8)
    return m

t0 = time.time()
IMGS = np.stack([load_gray(p) for p in META.path])
MASKS = [load_mask(mp) for mp in META.mask_paths]
HAS_MASK = np.array([m is not None for m in MASKS])
MASK_ARR = np.stack([m if m is not None else np.zeros((S, S), np.uint8) for m in MASKS])
LABELS = META.label.values.astype(np.float32)
SITE = META.site.values
print(f"Dimuat {len(IMGS)} citra dalam {time.time()-t0:.1f}s")

In [ ]:
# ===================== 4. PRA-PROSES =====================
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

def apply_clahe(arr):
    return np.stack([_clahe.apply(a) for a in arr])

def pooled_cdf(arr):
    h = np.bincount(arr.ravel(), minlength=256).astype(np.float64)
    c = np.cumsum(h); return c / c[-1]

def hist_match(arr, ref_cdf):
    """Cocokkan histogram tiap citra sumber ke CDF gabungan domain target (tanpa label)."""
    out = np.empty_like(arr)
    for i, a in enumerate(arr):
        c = pooled_cdf(a)
        lut = np.searchsorted(ref_cdf, c).clip(0, 255).astype(np.uint8)
        out[i] = lut[a]
    return out

def preprocess(arm, src_idx, tgt_idx):
    """Mengembalikan array citra (seluruh dataset) setelah pra-proses sesuai lengan."""
    X = IMGS.copy()
    if arm == "clahe":
        X = apply_clahe(X)                  # diterapkan ke sumber DAN target
    elif arm == "histmatch":
        ref = pooled_cdf(IMGS[tgt_idx])     # hanya intensitas target, tanpa label
        X[src_idx] = hist_match(IMGS[src_idx], ref)
    return X

MEAN, STD = np.array([0.485, 0.456, 0.406]), np.array([0.229, 0.224, 0.225])

def to_tensor(X):
    x = torch.from_numpy(X).float().div(255.).unsqueeze(1).repeat(1, 3, 1, 1)
    m = torch.tensor(MEAN).view(1, 3, 1, 1).float(); s = torch.tensor(STD).view(1, 3, 1, 1).float()
    return (x - m) / s

def augment(x):
    """Augmentasi ringan di GPU: random resized crop + rotasi kecil. Tanpa flip (anatomi CXR)."""
    B = x.size(0)
    ang = (torch.rand(B, device=x.device) - 0.5) * 2 * math.radians(7)
    sc = 1.0 / (0.85 + 0.15 * torch.rand(B, device=x.device))
    tx = (torch.rand(B, device=x.device) - 0.5) * 0.1
    ty = (torch.rand(B, device=x.device) - 0.5) * 0.1
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    x = F.grid_sample(x, grid, padding_mode="border", align_corners=False)
    # jitter kontras/kecerahan kecil
    a = 1 + (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.2
    b = (torch.rand(B, 1, 1, 1, device=x.device) - 0.5) * 0.2
    return x * a + b

In [ ]:
# ===================== 5. MODEL, MIXSTYLE =====================
class MixStyle(nn.Module):
    """Zhou et al. (ICLR 2021). Aktif hanya saat training."""
    def __init__(self, p=0.5, alpha=0.1, eps=1e-6):
        super().__init__(); self.p, self.alpha, self.eps = p, alpha, eps
        self.beta = torch.distributions.Beta(alpha, alpha); self.on = True
    def forward(self, x):
        if not (self.training and self.on) or random.random() > self.p or x.size(0) < 2:
            return x
        B = x.size(0)
        mu = x.mean([2, 3], keepdim=True); var = x.var([2, 3], keepdim=True)
        sig = (var + self.eps).sqrt(); mu, sig = mu.detach(), sig.detach()
        xn = (x - mu) / sig
        lam = self.beta.sample((B, 1, 1, 1)).to(x.device)
        perm = torch.randperm(B, device=x.device)
        mu2, sig2 = mu[perm], sig[perm]
        return xn * (lam * sig + (1 - lam) * sig2) + (lam * mu + (1 - lam) * mu2)

MIX_LAYERS = {"densenet121": ["features.denseblock1", "features.denseblock2"],
              "efficientnet_b0": ["blocks.1", "blocks.2"]}

def build_model(bb, mixstyle=False):
    m = timm.create_model(bb, pretrained=PRETRAINED, num_classes=1)
    m.mix = None
    if mixstyle:
        mix = MixStyle()
        for name in MIX_LAYERS[bb]:              # ganti layer X dengan Sequential(X, MixStyle)
            parent_name, child = name.rsplit(".", 1)
            parent = m.get_submodule(parent_name)
            setattr(parent, child, nn.Sequential(getattr(parent, child), mix))
        m.mix = mix                              # satu instans dipakai bersama (aman di-deepcopy)
    return m.to(DEVICE)

def predict(model, X, bs=64):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = X[i:i+bs].to(DEVICE)
            out.append(torch.sigmoid(model(xb)).squeeze(1).float().cpu())
    return torch.cat(out).numpy()

def safe_auc(y, p):
    if len(np.unique(y)) != 2 or not np.all(np.isfinite(p)):
        return np.nan
    return roc_auc_score(y, p)

In [ ]:
# ===================== 6. TRAINING SUMBER =====================
def train_supervised(model, Xtr, ytr, Xva=None, yva=None, epochs=20, lr=3e-4, patience=6, bs=32):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=CFG["wd"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    pos = ytr.mean().item(); pw = torch.tensor([(1 - pos) / max(pos, 1e-6)], device=DEVICE)
    lossf = nn.BCEWithLogitsLoss(pos_weight=pw)
    best, best_state, wait = -1, None, 0
    n = len(Xtr)
    for ep in range(epochs):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i+bs]
            if len(idx) < 2: continue
            xb = augment(Xtr[idx].to(DEVICE)); yb = ytr[idx].to(DEVICE)
            loss = lossf(model(xb).squeeze(1), yb)
            if not torch.isfinite(loss): continue
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
        sched.step()
        if Xva is not None:
            auc = safe_auc(yva.numpy(), predict(model, Xva))
            if auc > best + 1e-4:
                best, best_state, wait = auc, copy.deepcopy(model.state_dict()), 0
            else:
                wait += 1
                if wait >= patience: break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best

In [ ]:
# ===================== 7. METODE ADAPTASI =====================
def bn_layers(model):
    return [m for m in model.modules() if isinstance(m, nn.BatchNorm2d)]

def adabn(model, Xt, bs=64):
    """AdaBN: hitung ulang statistik BN di domain target tanpa label."""
    m = copy.deepcopy(model); m.eval()
    if m.mix is not None: m.mix.on = False
    for bn in bn_layers(m):
        bn.reset_running_stats(); bn.momentum = None; bn.train()
    with torch.no_grad():
        for i in range(0, len(Xt), bs):
            m(Xt[i:i+bs].to(DEVICE))
    m.eval(); return m

def binary_entropy(logit):
    p = torch.sigmoid(logit)
    return -(p * torch.log(p + 1e-6) + (1 - p) * torch.log(1 - p + 1e-6))

def tent(model, Xt, steps=1, bs=32, lr=1e-4):
    """TENT (Wang et al., ICLR 2021): minimasi entropi, hanya parameter afine BN, statistik batch target."""
    m = copy.deepcopy(model)
    if m.mix is not None: m.mix.on = False
    bn_only_train(m)
    for p in m.parameters(): p.requires_grad_(False)
    params = []
    for bn in bn_layers(m):
        bn.track_running_stats = False; bn.running_mean = None; bn.running_var = None
        bn.weight.requires_grad_(True); bn.bias.requires_grad_(True); params += [bn.weight, bn.bias]
    opt = torch.optim.Adam(params, lr=lr, betas=(0.9, 0.999))
    for _ in range(steps):
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i+bs]
            if len(idx) < 2: continue
            loss = binary_entropy(m(Xt[idx].to(DEVICE)).squeeze(1)).mean()
            if not torch.isfinite(loss): continue          # pengaman: lewati langkah divergen
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 5.0); opt.step()
    bn_only_train(m)   # TENT memprediksi memakai statistik batch target
    return m

def bn_only_train(model):
    """Mode eval untuk dropout/drop-path, tetapi BN memakai statistik batch (TENT)."""
    model.eval()
    for bn in bn_layers(model): bn.train()

def predict_tent(model, X, bs=64):
    bn_only_train(model); out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            out.append(torch.sigmoid(model(X[i:i+bs].to(DEVICE))).squeeze(1).cpu())
    return torch.cat(out).numpy()

def shot(model, Xt, epochs=5, beta=0.3, bs=32, lr=1e-4):
    """SHOT (Liang et al., ICML 2020): head dibekukan; encoder dilatih dengan information
    maximization + pseudo-label berbasis centroid."""
    m = copy.deepcopy(model)
    if m.mix is not None: m.mix.on = False
    head = m.get_classifier()
    for p in head.parameters(): p.requires_grad_(False)
    enc_params = [p for n, p in m.named_parameters() if p.requires_grad]
    opt = torch.optim.SGD(enc_params, lr=lr * 5, momentum=0.9, weight_decay=1e-3)

    def feats_logits(x):
        f = m.forward_head(m.forward_features(x), pre_logits=True)
        return f, head(f).squeeze(1)

    for ep in range(epochs):
        # pseudo-label centroid (dihitung ulang tiap epoch)
        m.eval(); fs, ps = [], []
        with torch.no_grad():
            for i in range(0, len(Xt), 64):
                f, lg = feats_logits(Xt[i:i+64].to(DEVICE))
                fs.append(F.normalize(f, dim=1)); ps.append(torch.sigmoid(lg))
        fs = torch.cat(fs); ps = torch.cat(ps)
        P = torch.stack([1 - ps, ps], 1)                       # N x 2
        cent = F.normalize(P.t() @ fs / (P.sum(0, keepdim=True).t() + 1e-6), dim=1)
        for _ in range(2):                                     # satu putaran k-means
            pl = (fs @ cent.t()).argmax(1)
            oh = F.one_hot(pl, 2).float()
            cent = F.normalize(oh.t() @ fs / (oh.sum(0, keepdim=True).t() + 1e-6), dim=1)
        pl = (fs @ cent.t()).argmax(1).float()
        m.train()
        for bn in bn_layers(m): bn.eval()                       # stabil untuk batch kecil
        perm = torch.randperm(len(Xt))
        for i in range(0, len(Xt), bs):
            idx = perm[i:i+bs]
            if len(idx) < 2: continue
            _, lg = feats_logits(Xt[idx].to(DEVICE))
            p = torch.sigmoid(lg)
            ent = binary_entropy(lg).mean()
            pm = p.mean(); div = -(pm * torch.log(pm + 1e-6) + (1 - pm) * torch.log(1 - pm + 1e-6))
            ce = F.binary_cross_entropy_with_logits(lg, pl[idx].to(DEVICE))
            loss = ent - div + beta * ce
            if not torch.isfinite(loss): continue          # pengaman: lewati langkah divergen
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(enc_params, 5.0); opt.step()
    m.eval(); return m

def few_shot(model, Xs, ys, epochs=30, lr=5e-5):
    m = copy.deepcopy(model)
    if m.mix is not None: m.mix.on = False
    m, _ = train_supervised(m, Xs, ys, None, None, epochs=epochs, lr=lr, bs=min(32, len(Xs)))
    m.eval(); return m

In [ ]:
# ===================== 8. METRIK =====================
def sens_at_spec(y, p, spec=0.70):
    fpr, tpr, _ = roc_curve(y, p); ok = fpr <= 1 - spec
    return tpr[ok].max() if ok.any() else 0.0

def spec_at_sens(y, p, sens=0.90):
    fpr, tpr, _ = roc_curve(y, p); ok = tpr >= sens
    return (1 - fpr[ok]).max() if ok.any() else 0.0

def ece(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1); e = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        s = (p >= lo) & (p < hi) if hi < 1 else (p >= lo) & (p <= hi)
        if s.any(): e += s.mean() * abs(y[s].mean() - p[s].mean())
    return e

def metrics(y, p):
    yhat = (p >= 0.5).astype(int)
    tpr = ((yhat == 1) & (y == 1)).sum() / max((y == 1).sum(), 1)
    tnr = ((yhat == 0) & (y == 0)).sum() / max((y == 0).sum(), 1)
    return dict(auc=safe_auc(y, p), bacc=(tpr + tnr) / 2, sens05=tpr, spec05=tnr,
                sens_at_spec70=sens_at_spec(y, p), spec_at_sens90=spec_at_sens(y, p), ece=ece(y, p))

In [ ]:
# ===================== 9. AUDIT PENJELASAN: GRAD-CAM, LAR, OKLUSI =====================
def gradcam(model, X, tent_mode=False, bs=16):
    """Grad-CAM pada peta fitur terakhir (forward_features) untuk kelas yang diprediksi."""
    bn_only_train(model) if tent_mode else model.eval()
    cams = []
    for i in range(0, len(X), bs):
        xb = X[i:i+bs].to(DEVICE)
        with torch.enable_grad():
            fmap = model.forward_features(xb); fmap.retain_grad()
            logit = model.forward_head(fmap).squeeze(1)
            sign = torch.where(logit >= 0, 1.0, -1.0)            # kelas yang diprediksi
            (logit * sign).sum().backward()
            w = fmap.grad.mean((2, 3), keepdim=True)
            cam = F.relu((w * fmap).sum(1, keepdim=True))
            cam = F.interpolate(cam, size=(S, S), mode="bilinear", align_corners=False)
        cams.append(cam.squeeze(1).detach().cpu())
        model.zero_grad(set_to_none=True)
    return torch.cat(cams).numpy()

def lar_stats(cams, masks):
    tot = cams.reshape(len(cams), -1).sum(1)
    inside = (cams * masks).reshape(len(cams), -1).sum(1)
    lar = np.where(tot > 0, inside / np.maximum(tot, 1e-12), np.nan)
    area = masks.reshape(len(masks), -1).mean(1)
    return lar, lar / np.maximum(area, 1e-6)

_BLUR = lambda a: cv2.GaussianBlur(a, (0, 0), sigmaX=15)

def occlude(Xraw, masks, where):
    """where='outside' blur di luar paru; 'lung' blur di dalam paru."""
    out = Xraw.copy()
    for i in range(len(out)):
        b = _BLUR(out[i]); mk = cv2.dilate(masks[i], np.ones((7, 7), np.uint8)).astype(bool)
        sel = ~mk if where == "outside" else mk
        out[i][sel] = b[sel]
    return out

In [ ]:
# ===================== 10. LOOP UTAMA =====================
PRED_ROWS, MET_ROWS, CAM_ROWS = [], [], []
done_path = os.path.join(OUT_DIR, "metrics.csv")

ERR_ROWS = []

def log(method, ctx, y, p, keys, extra=None):
    row = dict(ctx); row.update(method=method)
    n_bad = int((~np.isfinite(p)).sum())
    if n_bad:
        ERR_ROWS.append(dict(ctx, method=method, stage="predict", error=f"{n_bad} prediksi NaN/inf"))
        print(f"   ! {method}: {n_bad} prediksi non-finit -> dicatat sebagai gagal")
        row.update(nan_preds=n_bad); MET_ROWS.append(row); return
    row.update(metrics(y, p))
    if extra: row.update(extra)
    MET_ROWS.append(row)
    for k, yy, pp in zip(keys, y, p):
        PRED_ROWS.append(dict(ctx, method=method, key=k, label=int(yy), prob=float(pp)))

def audit(method, ctx, model, Xt_tensor, Xt_raw, y, keys, has_m, masks, tent_mode=False):
    """LAR + oklusi untuk citra target yang punya masker."""
    if has_m.sum() < 4: return {}
    idx = np.where(has_m)[0]
    cams = gradcam(model, Xt_tensor[idx], tent_mode=tent_mode, bs=CFG["cam_batch"])
    lar, enr = lar_stats(cams, masks[idx])
    for k, l, e, yy in zip(np.array(keys)[idx], lar, enr, y[idx]):
        CAM_ROWS.append(dict(ctx, method=method, key=k, label=int(yy), lar=l, enrichment=e))
    pf = predict_tent if tent_mode else predict
    Xo = to_tensor(occlude(Xt_raw[idx], masks[idx], "outside"))
    Xl = to_tensor(occlude(Xt_raw[idx], masks[idx], "lung"))
    return dict(lar_mean=np.nanmean(lar), lar_median=np.nanmedian(lar), enrich_mean=np.nanmean(enr),
                auc_masked_subset=safe_auc(y[idx], pf(model, Xt_tensor[idx])),
                auc_outside_blurred=safe_auc(y[idx], pf(model, Xo)),
                auc_lung_occluded=safe_auc(y[idx], pf(model, Xl)))

def run_config(bb, src, tgt, arm, seed):
    set_seed(seed)
    ctx = dict(backbone=bb, direction=f"{src}->{tgt}", arm=arm, seed=seed)
    src_idx = np.where(SITE == src)[0]; tgt_idx = np.where(SITE == tgt)[0]
    X_all = preprocess("baseline" if arm == "mixstyle" else arm, src_idx, tgt_idx)

    # --- split sumber train/val
    tr, va = train_test_split(src_idx, test_size=0.2, stratify=LABELS[src_idx], random_state=seed)
    # --- target: pool berlabel (few-shot) vs test
    rng = np.random.RandomState(seed); pool = []
    for c in [0, 1]:
        cand = tgt_idx[LABELS[tgt_idx] == c]
        pool += list(rng.choice(cand, CFG["pool_per_class"], replace=False))
    pool = np.array(pool); test = np.setdiff1d(tgt_idx, pool)

    Xtr, ytr = to_tensor(X_all[tr]), torch.tensor(LABELS[tr])
    Xva, yva = to_tensor(X_all[va]), torch.tensor(LABELS[va])
    Xte_raw = X_all[test]; Xte = to_tensor(Xte_raw); yte = LABELS[test]
    keys = META.key.values[test]; hm = HAS_MASK[test]; mk = MASK_ARR[test]

    model = build_model(bb, mixstyle=(arm == "mixstyle"))
    model, val_auc = train_supervised(model, Xtr, ytr, Xva, yva, epochs=CFG["epochs"],
                                      lr=CFG["lr"], patience=CFG["patience"], bs=CFG["batch"])
    if model.mix is not None: model.mix.on = False
    ctx["src_val_auc"] = val_auc

    def step(name, make_model, pred_fn=predict, tent_mode=False):
        """Setiap metode diisolasi: kegagalan satu metode tidak menghapus metode lain."""
        try:
            mm = make_model()
            p = pred_fn(mm, Xte)
        except Exception as e:
            ERR_ROWS.append(dict(ctx, method=name, stage="adapt", error=repr(e)[:300]))
            print(f"   ! {name} gagal saat adaptasi: {e}"); return
        try:
            ex = audit(name, ctx, mm, Xte, Xte_raw, yte, keys, hm, mk, tent_mode=tent_mode)
        except Exception as e:
            ERR_ROWS.append(dict(ctx, method=name, stage="audit", error=repr(e)[:300]))
            print(f"   ! {name} gagal saat audit: {e}"); ex = {}
        log(name, ctx, yte, p, keys, ex)
        del mm

    step("source_only", lambda: model)
    step("adabn", lambda: adabn(model, Xte))
    step("tent", lambda: tent(model, Xte, steps=CFG["tent_steps"]), pred_fn=predict_tent, tent_mode=True)
    step("shot", lambda: shot(model, Xte, epochs=CFG["shot_epochs"], beta=CFG["shot_beta"]))
    # Few-shot (hanya untuk lengan baseline & mixstyle agar hemat waktu)
    if arm in ("baseline", "mixstyle"):
        for k in CFG["shots"]:
            sup = np.concatenate([pool[LABELS[pool] == c][:k] for c in [0, 1]])
            step(f"fewshot_k{k}", lambda sup=sup: few_shot(model, to_tensor(X_all[sup]), torch.tensor(LABELS[sup]),
                                                          epochs=CFG["fs_epochs"], lr=CFG["fs_lr"]))
    del model
    if DEVICE == "cuda": torch.cuda.empty_cache()

def run_oracle(bb, tgt, seed):
    """Batas atas: 5-fold CV di dalam domain target (pra-proses baseline)."""
    set_seed(seed)
    tgt_idx = np.where(SITE == tgt)[0]; X_all = IMGS
    skf = StratifiedKFold(CFG["oracle_folds"], shuffle=True, random_state=seed)
    oof = np.zeros(len(tgt_idx))
    for trf, tef in skf.split(tgt_idx, LABELS[tgt_idx]):
        tr_i = tgt_idx[trf]
        tr2, va2 = train_test_split(tr_i, test_size=0.2, stratify=LABELS[tr_i], random_state=seed)
        m = build_model(bb)
        m, _ = train_supervised(m, to_tensor(X_all[tr2]), torch.tensor(LABELS[tr2]),
                                to_tensor(X_all[va2]), torch.tensor(LABELS[va2]),
                                epochs=CFG["epochs"], lr=CFG["lr"], patience=CFG["patience"], bs=CFG["batch"])
        oof[tef] = predict(m, to_tensor(X_all[tgt_idx[tef]])); del m
    ctx = dict(backbone=bb, direction=f"oracle->{tgt}", arm="oracle_cv", seed=seed, src_val_auc=np.nan)
    log("oracle_cv", ctx, LABELS[tgt_idx], oof, META.key.values[tgt_idx])

def save_all():
    pd.DataFrame(MET_ROWS).to_csv(os.path.join(OUT_DIR, "metrics.csv"), index=False)
    pd.DataFrame(PRED_ROWS).to_csv(os.path.join(OUT_DIR, "predictions.csv"), index=False)
    pd.DataFrame(CAM_ROWS).to_csv(os.path.join(OUT_DIR, "lar_per_image.csv"), index=False)
    pd.DataFrame(ERR_ROWS).to_csv(os.path.join(OUT_DIR, "errors.csv"), index=False)

T0 = time.time()
jobs = [(bb, s, t, arm, sd) for bb in CFG["backbones"] for (s, t) in CFG["directions"]
        for arm in CFG["arms"] for sd in CFG["seeds"]]
print(f"Total konfigurasi: {len(jobs)}")
for j, (bb, s, t, arm, sd) in enumerate(jobs, 1):
    t1 = time.time()
    try:
        run_config(bb, s, t, arm, sd)
    except Exception as e:
        print(f"!! GAGAL {bb} {s}->{t} {arm} seed{sd}: {e}")
    save_all()
    last = [r for r in MET_ROWS if r.get("backbone") == bb and r.get("arm") == arm and r.get("seed") == sd
            and r.get("direction") == f"{s}->{t}" and r["method"] == "source_only"]
    auc0 = last[-1]["auc"] if last else float("nan")
    print(f"[{j}/{len(jobs)}] {bb:16s} {s}->{t} {arm:9s} seed{sd} | src-only AUC={auc0:.3f} "
          f"| {time.time()-t1:.0f}s | total {(time.time()-T0)/60:.1f} min")

for bb in CFG["backbones"]:
    for t in ["MC", "SZ"]:
        for sd in CFG["oracle_seeds"]:
            try: run_oracle(bb, t, sd)
            except Exception as e: print(f"!! Oracle gagal {bb} {t}: {e}")
            save_all()
print(f"Selesai dalam {(time.time()-T0)/60:.1f} menit")
print(f"Jumlah kegagalan metode: {len(ERR_ROWS)}  (rincian di results/errors.csv)")
if ERR_ROWS:
    print(pd.DataFrame(ERR_ROWS).groupby(["backbone", "method", "stage"]).size().to_string())

In [ ]:
# ===================== 11. ANALISIS & TABEL =====================
def delong_roc_variance(ground_truth, predictions):
    """Fast DeLong (Sun & Xu, 2014) untuk satu atau beberapa prediktor berpasangan."""
    order = (-ground_truth).argsort(); label_1_count = int(ground_truth.sum())
    preds = predictions[:, order]
    m, n = label_1_count, preds.shape[1] - label_1_count
    pos, neg = preds[:, :m], preds[:, m:]
    def midrank(x):
        J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
        while i < N:
            j = i
            while j < N and Z[j] == Z[i]: j += 1
            T[i:j] = 0.5 * (i + j - 1); i = j
        T2 = np.empty(N); T2[J] = T + 1; return T2
    k = preds.shape[0]
    tx = np.array([midrank(r) for r in pos]); ty = np.array([midrank(r) for r in neg])
    tz = np.array([midrank(r) for r in preds])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n; v10 = 1.0 - (tz[:, m:] - ty) / m
    sx = np.atleast_2d(np.cov(v01)); sy = np.atleast_2d(np.cov(v10))
    return aucs, sx / m + sy / n

def delong_test(y, p1, p2):
    aucs, cov = delong_roc_variance(np.asarray(y), np.vstack([p1, p2]))
    l = np.array([[1, -1]]); z = np.abs(np.diff(aucs)) / np.sqrt(max((l @ cov @ l.T).item(), 1e-12))
    return float(aucs[1] - aucs[0]), float(2 * stats.norm.sf(z).item())

met = pd.read_csv(os.path.join(OUT_DIR, "metrics.csv"))
if "nan_preds" in met.columns:
    print("Baris gagal (prediksi NaN) dikeluarkan dari analisis:", int(met.nan_preds.notna().sum()))
    met = met[met.nan_preds.isna()].copy()
pred = pd.read_csv(os.path.join(OUT_DIR, "predictions.csv"))

# Tabel 1: ringkasan per arah × backbone × lengan × metode (mean ± sd lintas seed)
cols = ["auc", "sens_at_spec70", "spec_at_sens90", "ece", "lar_mean", "enrich_mean",
        "auc_outside_blurred", "auc_lung_occluded"]
cols = [c for c in cols if c in met.columns]
g = met.groupby(["direction", "backbone", "arm", "method"])[cols]
summary = g.mean().round(3).astype(str) + " ± " + g.std().fillna(0).round(3).astype(str)
summary.to_csv(os.path.join(OUT_DIR, "table_summary.csv"))
try:
    display(summary)
except NameError:
    print(summary.to_string())

# Tabel 2: ΔAUC vs ΔLAR terhadap source-only (konfigurasi berpasangan)
base = met[met.method == "source_only"].set_index(["direction", "backbone", "arm", "seed"])
d = met[~met.method.isin(["source_only", "oracle_cv"])].copy()
d = d.join(base[["auc", "lar_mean", "auc_lung_occluded"]].add_suffix("_src"),
           on=["direction", "backbone", "arm", "seed"])
d["dAUC"] = d.auc - d.auc_src; d["dLAR"] = d.lar_mean - d.lar_mean_src
d["dLungOcclAUC"] = d.auc_lung_occluded - d.auc_lung_occluded_src
d.to_csv(os.path.join(OUT_DIR, "deltas.csv"), index=False)
ok = d[["dAUC", "dLAR"]].dropna()
if len(ok) > 3:
    rho, pv = stats.spearmanr(ok.dAUC, ok.dLAR)
    print(f"\nSpearman ΔAUC vs ΔLAR: rho={rho:.3f}, p={pv:.4f} (n={len(ok)})")
    # 'Higher AUC, wrong reasons' = AUC naik tetapi LAR turun
    wr = ((d.dAUC > 0.02) & (d.dLAR < -0.02)).groupby([d.direction, d.method]).mean().round(3)
    print("\nProporsi konfigurasi 'AUC naik, LAR turun':\n", wr)
    wr.to_csv(os.path.join(OUT_DIR, "wrong_reasons_rate.csv"))

# Tabel 3: Wilcoxon antar-seed (metode vs source-only) + DeLong per seed
rows = []
for (dr, bb, arm, mth), sub in d.groupby(["direction", "backbone", "arm", "method"]):
    w = stats.wilcoxon(sub.auc, sub.auc_src).pvalue if len(sub) >= 5 and (sub.auc != sub.auc_src).any() else np.nan
    dl = []
    for sd in sub.seed:
        a = pred[(pred.direction == dr) & (pred.backbone == bb) & (pred.arm == arm) & (pred.seed == sd)]
        p0 = a[a.method == "source_only"].sort_values("key"); p1 = a[a.method == mth].sort_values("key")
        if len(p0) == len(p1) and len(p0) > 0:
            dl.append(delong_test(p0.label.values, p0.prob.values, p1.prob.values)[1])
    rows.append(dict(direction=dr, backbone=bb, arm=arm, method=mth, n_seeds=len(sub),
                     dAUC_mean=sub.dAUC.mean(), wilcoxon_p=w,
                     delong_p_median=np.median(dl) if dl else np.nan,
                     delong_sig_frac=np.mean(np.array(dl) < 0.05) if dl else np.nan))
tests = pd.DataFrame(rows).round(4)
tests.to_csv(os.path.join(OUT_DIR, "table_tests.csv"), index=False)
print(tests.to_string())

In [ ]:
# ===================== 12. GAMBAR =====================
import matplotlib.pyplot as plt
FIG = os.path.join(OUT_DIR, "figures"); os.makedirs(FIG, exist_ok=True)

# Gambar 1: ΔAUC vs ΔLAR
if len(ok) > 3:
    fig, ax = plt.subplots(figsize=(6.5, 5))
    for mth, sub in d.dropna(subset=["dAUC", "dLAR"]).groupby("method"):
        ax.scatter(sub.dLAR, sub.dAUC, label=mth, s=22, alpha=0.75)
    ax.axhline(0, c="grey", lw=0.8); ax.axvline(0, c="grey", lw=0.8)
    ax.set_xlabel("Δ Lung Attention Ratio vs source-only"); ax.set_ylabel("Δ AUC vs source-only")
    ax.set_title("Higher AUC, wrong reasons? (kuadran kiri atas)")
    ax.legend(fontsize=8, frameon=False); fig.tight_layout()
    fig.savefig(os.path.join(FIG, "fig_dauc_vs_dlar.png"), dpi=300); plt.show()

# Gambar 2: kurva efisiensi label (few-shot)
fs = met[met.method.str.startswith("fewshot") | (met.method == "source_only")].copy()
fs = fs[fs.arm.isin(["baseline", "mixstyle"])]
fs["k"] = fs.method.str.extract(r"k(\d+)")[0].astype(float).fillna(0)
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (dr, sub) in zip(axes, fs.groupby("direction")):
    for (bb, arm), s2 in sub.groupby(["backbone", "arm"]):
        agg = s2.groupby("k").auc.agg(["mean", "std"])
        ax.errorbar(agg.index, agg["mean"], yerr=agg["std"], marker="o", capsize=3, label=f"{bb}/{arm}")
    orc = met[(met.method == "oracle_cv") & (met.direction == f"oracle->{dr.split('->')[1]}")]
    if len(orc): ax.axhline(orc.auc.mean(), ls="--", c="k", lw=1, label="oracle CV")
    ax.set_title(dr); ax.set_xlabel("citra berlabel target per kelas (k)"); ax.set_xticks([0, 5, 10, 20])
axes[0].set_ylabel("AUC target"); axes[-1].legend(fontsize=7, frameon=False)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "fig_label_efficiency.png"), dpi=300); plt.show()

# Gambar 3: uji oklusi (AUC asli vs luar-paru-diblur vs paru-diblur)
occ = met[met.method != "oracle_cv"].groupby(["direction", "method"])[["auc_masked_subset", "auc_outside_blurred", "auc_lung_occluded"]].mean()
occ.to_csv(os.path.join(OUT_DIR, "table_occlusion.csv"))
for dr, sub in occ.groupby(level=0):
    sub = sub.droplevel(0)
    ax = sub.plot.bar(figsize=(8, 3.8), rot=30); ax.axhline(0.5, ls=":", c="grey")
    ax.set_title(f"Uji oklusi — {dr}"); ax.set_ylabel("AUC"); plt.tight_layout()
    plt.savefig(os.path.join(FIG, f"fig_occlusion_{dr.replace('->', '_to_')}.png"), dpi=300); plt.show()

In [ ]:
# ===================== 13. CONTOH VISUAL GRAD-CAM =====================
# Melatih ulang satu model (densenet121, SZ->MC, baseline, seed 0) untuk ilustrasi
# perbandingan peta atensi source-only vs SHOT vs few-shot k=20 pada 4 citra TB.
try:
    bb, src, tgt, sd = CFG["backbones"][0], "SZ", "MC", CFG["seeds"][0]
    set_seed(sd)
    src_idx = np.where(SITE == src)[0]; tgt_idx = np.where(SITE == tgt)[0]
    tr, va = train_test_split(src_idx, test_size=0.2, stratify=LABELS[src_idx], random_state=sd)
    mdl = build_model(bb)
    mdl, _ = train_supervised(mdl, to_tensor(IMGS[tr]), torch.tensor(LABELS[tr]),
                              to_tensor(IMGS[va]), torch.tensor(LABELS[va]),
                              epochs=CFG["epochs"], lr=CFG["lr"], patience=CFG["patience"])
    show = tgt_idx[(LABELS[tgt_idx] == 1) & HAS_MASK[tgt_idx]][:4]
    Xs = to_tensor(IMGS[show])
    mods = {"source-only": mdl, "SHOT": shot(mdl, to_tensor(IMGS[tgt_idx]), epochs=CFG["shot_epochs"])}
    fig, axes = plt.subplots(len(mods), len(show), figsize=(3 * len(show), 3 * len(mods)))
    for r, (nm, mm) in enumerate(mods.items()):
        cams = gradcam(mm, Xs)
        for c in range(len(show)):
            a = axes[r, c]; a.imshow(IMGS[show[c]], cmap="gray")
            a.imshow(cams[c] / (cams[c].max() + 1e-8), cmap="jet", alpha=0.4)
            a.contour(MASK_ARR[show[c]], levels=[0.5], colors="w", linewidths=0.8)
            lar, _ = lar_stats(cams[c:c+1], MASK_ARR[show[c]:show[c]+1])
            a.set_title(f"{nm} | LAR={lar[0]:.2f}", fontsize=9); a.axis("off")
    plt.tight_layout(); plt.savefig(os.path.join(FIG, "fig_gradcam_examples.png"), dpi=300); plt.show()
except Exception as e:
    print("Contoh Grad-CAM dilewati:", e)

print("\nFile keluaran:")
for p in sorted(glob.glob(os.path.join(OUT_DIR, "**", "*"), recursive=True)):
    print(" ", p)